# Week 1: Data Validation and Exploratory Data Analysis

## Project Objective

This project focuses on multi-touch marketing attribution. The aim is to understand how different marketing touchpoints contribute to customer purchases.

The available datasets include web events, transactions, customers, products, and campaign metadata.

The current phase focuses on:
- Dataset validation
- Customer journey analysis
- Funnel analysis
- First-touch attribution preparation
- Last-touch attribution preparation
- Linear attribution preparation

## Data Limitation

The shared datasets do not currently include ad spend, campaign cost, budget, CPC, or CPM data.

Therefore, true ROAS, CAC, CPC, and ROI cannot be calculated directly at this stage. These metrics will require an additional ad_spend dataset or a synthetic ad_spend table later.

In [47]:
import pandas as pd
import numpy as np
from pathlib import Path

In [48]:
DATA_DIR = Path("../data")

events_path = DATA_DIR / "events.csv"
transactions_path = DATA_DIR / "transactions.csv"
customers_path = DATA_DIR / "customers.csv"
products_path = DATA_DIR / "products.csv"
campaigns_path = DATA_DIR / "campaigns.csv"

required_files = {
    "events": events_path,
    "transactions": transactions_path,
    "customers": customers_path,
    "products": products_path,
    "campaigns": campaigns_path
}

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'} - {path}")

events: FOUND - ..\data\events.csv
transactions: FOUND - ..\data\transactions.csv
customers: FOUND - ..\data\customers.csv
products: FOUND - ..\data\products.csv
campaigns: FOUND - ..\data\campaigns.csv


In [49]:
events = pd.read_csv(events_path, low_memory=False)
transactions = pd.read_csv(transactions_path, low_memory=False)
customers = pd.read_csv(customers_path, low_memory=False)
products = pd.read_csv(products_path, low_memory=False)
campaigns = pd.read_csv(campaigns_path, low_memory=False)

print("Data loaded successfully.")

Data loaded successfully.


In [50]:
datasets = {
    "events": events,
    "transactions": transactions,
    "customers": customers,
    "products": products,
    "campaigns": campaigns
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows and {df.shape[1]} columns")

events: 2,000,000 rows and 12 columns
transactions: 103,127 rows and 9 columns
customers: 100,000 rows and 7 columns
products: 2,000 rows and 6 columns
campaigns: 50 rows and 7 columns


In [51]:
for name, df in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


EVENTS COLUMNS
['event_id', 'timestamp', 'customer_id', 'session_id', 'event_type', 'product_id', 'device_type', 'traffic_source', 'campaign_id', 'page_category', 'session_duration_sec', 'experiment_group']

TRANSACTIONS COLUMNS
['transaction_id', 'timestamp', 'customer_id', 'product_id', 'quantity', 'discount_applied', 'gross_revenue', 'campaign_id', 'refund_flag']

CUSTOMERS COLUMNS
['customer_id', 'signup_date', 'country', 'age', 'gender', 'loyalty_tier', 'acquisition_channel']

PRODUCTS COLUMNS
['product_id', 'category', 'brand', 'base_price', 'launch_date', 'is_premium']

CAMPAIGNS COLUMNS
['campaign_id', 'channel', 'objective', 'start_date', 'end_date', 'target_segment', 'expected_uplift']


In [52]:
def missing_value_report(df):
    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().sum().values / len(df) * 100).round(2)
    })
    return report.sort_values("missing_count", ascending=False)

for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    display(missing_value_report(df))


Missing values in events:


,column,missing_count,missing_percent
5,product_id,200371,10.02
6,device_type,40300,2.02
0,event_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
3,session_id,0,0.00
4,event_type,0,0.00
7,traffic_source,0,0.00
8,campaign_id,0,0.00
9,page_category,0,0.00



Missing values in transactions:


,column,missing_count,missing_percent
3,product_id,10449,10.13
6,gross_revenue,10449,10.13
0,transaction_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
4,quantity,0,0.00
5,discount_applied,0,0.00
7,campaign_id,0,0.00
8,refund_flag,0,0.00



Missing values in customers:


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,signup_date,0,0.0
2,country,0,0.0
3,age,0,0.0
4,gender,0,0.0
5,loyalty_tier,0,0.0
6,acquisition_channel,0,0.0



Missing values in products:


,column,missing_count,missing_percent
0,product_id,0,0.0
1,category,0,0.0
2,brand,0,0.0
3,base_price,0,0.0
4,launch_date,0,0.0
5,is_premium,0,0.0



Missing values in campaigns:


,column,missing_count,missing_percent
0,campaign_id,0,0.0
1,channel,0,0.0
2,objective,0,0.0
3,start_date,0,0.0
4,end_date,0,0.0
5,target_segment,0,0.0
6,expected_uplift,0,0.0


In [53]:
spend_keywords = ["spend", "cost", "budget", "cpc", "cpm", "roas", "cac"]

for name, df in datasets.items():
    spend_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in spend_keywords)
    ]
    print(f"{name}: {spend_columns}")

events: []
transactions: []
customers: []
products: []
campaigns: []


## Ad Spend Data Availability Check

The current datasets include events, transactions, customers, products, and campaigns.

After checking all column names, no spend, cost, budget, CPC, CPM, ROAS, or CAC-related column was found.

Therefore, true ROAS and CAC cannot be calculated directly from the current data. The current analysis will focus on customer journey analysis, funnel metrics, attribution models, and attributed revenue. A synthetic ad spend table can be created later if approved by the team.

## Missing Value Analysis

This section checks missing values in each dataset. Missing values are important because they may affect joins, revenue calculation, attribution logic, and dashboard accuracy.

In [54]:
def missing_value_report(df):
    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().sum().values / len(df) * 100).round(2)
    })
    return report.sort_values("missing_count", ascending=False)

for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    display(missing_value_report(df))


Missing values in events:


,column,missing_count,missing_percent
5,product_id,200371,10.02
6,device_type,40300,2.02
0,event_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
3,session_id,0,0.00
4,event_type,0,0.00
7,traffic_source,0,0.00
8,campaign_id,0,0.00
9,page_category,0,0.00



Missing values in transactions:


,column,missing_count,missing_percent
3,product_id,10449,10.13
6,gross_revenue,10449,10.13
0,transaction_id,0,0.00
1,timestamp,0,0.00
2,customer_id,0,0.00
4,quantity,0,0.00
5,discount_applied,0,0.00
7,campaign_id,0,0.00
8,refund_flag,0,0.00



Missing values in customers:


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,signup_date,0,0.0
2,country,0,0.0
3,age,0,0.0
4,gender,0,0.0
5,loyalty_tier,0,0.0
6,acquisition_channel,0,0.0



Missing values in products:


,column,missing_count,missing_percent
0,product_id,0,0.0
1,category,0,0.0
2,brand,0,0.0
3,base_price,0,0.0
4,launch_date,0,0.0
5,is_premium,0,0.0



Missing values in campaigns:


,column,missing_count,missing_percent
0,campaign_id,0,0.0
1,channel,0,0.0
2,objective,0,0.0
3,start_date,0,0.0
4,end_date,0,0.0
5,target_segment,0,0.0
6,expected_uplift,0,0.0


## Missing Value Findings

The missing value report shows that some fields contain missing values.

In the events dataset, missing product_id values may occur because not every web event is related to a specific product. For example, home page views, checkout page visits, or bounce events may not always have a product_id.

In the transactions dataset, missing product_id and gross_revenue values are more important because they affect revenue and product-level analysis. Rows with missing gross_revenue should not be used for revenue attribution calculations.

For attribution modeling, only valid non-refunded transactions with available gross_revenue will be used.

## Timestamp and Date Conversion

Attribution depends on the correct order of customer events. Therefore, timestamp and date columns must be converted into proper datetime format.

In [55]:
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
products["launch_date"] = pd.to_datetime(products["launch_date"], errors="coerce")
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"], errors="coerce")
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"], errors="coerce")

print("Date conversion completed.")

Date conversion completed.


In [56]:
date_columns = {
    "events.timestamp": events["timestamp"],
    "transactions.timestamp": transactions["timestamp"],
    "customers.signup_date": customers["signup_date"],
    "products.launch_date": products["launch_date"],
    "campaigns.start_date": campaigns["start_date"],
    "campaigns.end_date": campaigns["end_date"]
}

for label, series in date_columns.items():
    print(f"\n{label}")
    print(f"Missing or invalid dates: {series.isna().sum():,}")
    print(f"Minimum date: {series.min()}")
    print(f"Maximum date: {series.max()}")


events.timestamp
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:01:28
Maximum date: 2023-12-31 23:57:50

transactions.timestamp
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:12:50
Maximum date: 2023-12-31 22:37:32

customers.signup_date
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:00:00
Maximum date: 2023-12-31 00:00:00

products.launch_date
Missing or invalid dates: 0
Minimum date: 2021-01-01 00:00:00
Maximum date: 2023-12-31 00:00:00

campaigns.start_date
Missing or invalid dates: 0
Minimum date: 2021-01-20 00:00:00
Maximum date: 2023-11-04 00:00:00

campaigns.end_date
Missing or invalid dates: 0
Minimum date: 2021-02-21 00:00:00
Maximum date: 2024-01-06 00:00:00


## Traffic Source Cleaning

The traffic_source column may contain inconsistent capitalization such as Organic and ORGANIC. This section standardizes traffic source values for accurate grouping.

In [57]:
events["traffic_source_clean"] = (
    events["traffic_source"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.title()
)

print("Original traffic source values:")
print(events["traffic_source"].value_counts())

print("\nCleaned traffic source values:")
print(events["traffic_source_clean"].value_counts())

Original traffic source values:
Organic        776758
Paid Search    387657
Social         291194
Email          290651
Direct         193870
ORGANIC         23731
PAID SEARCH     12181
SOCIAL           9077
EMAIL            8989
DIRECT           5892
Name: traffic_source, dtype: int64

Cleaned traffic source values:
Organic        800489
Paid Search    399838
Social         300271
Email          299640
Direct         199762
Name: traffic_source_clean, dtype: int64


## campaign_id = 0 Check

The campaigns table contains official campaign IDs. However, events and transactions may contain campaign_id = 0. This is likely organic, direct, no campaign, or unattributed activity.

In [58]:
print("Events with campaign_id = 0:", (events["campaign_id"] == 0).sum())
print("Transactions with campaign_id = 0:", (transactions["campaign_id"] == 0).sum())
print("Campaign ID range in campaigns table:", campaigns["campaign_id"].min(), "to", campaigns["campaign_id"].max())

Events with campaign_id = 0: 1000251
Transactions with campaign_id = 0: 20955
Campaign ID range in campaigns table: 1 to 50


# Week 3: Metric Calculation and Dashboard-Ready Tables

In this section, I prepare business metrics and summary tables for the dashboard.

The focus is on:
- Revenue metrics
- Refund-adjusted revenue
- Funnel metrics
- Attribution summary by channel
- Attribution summary by campaign
- Customer segment performance
- Product category performance

Since ad spend data is not available yet, these metrics are revenue-based. ROAS and CAC will be added later after creating or receiving ad spend data.

campaign_id = 0 will be treated as "No Campaign / Organic / Direct / Unattributed" instead of being treated as an error. This helps preserve unattributed customer activity during analysis.

In [59]:
# Create an extra campaign row for campaign_id = 0

no_campaign_row = pd.DataFrame([{
    "campaign_id": 0,
    "channel": "No Campaign",
    "objective": "Organic/Direct/Unattributed",
    "start_date": pd.NaT,
    "end_date": pd.NaT,
    "target_segment": "Unknown",
    "expected_uplift": 0
}])

campaigns_extended = pd.concat(
    [campaigns, no_campaign_row],
    ignore_index=True
)

campaigns_extended.tail()

,campaign_id,channel,objective,start_date,end_date,target_segment,expected_uplift
46,47,Affiliate,Cross-sell,2022-04-12,2022-06-04,High Value,0.066
47,48,Paid Search,Acquisition,2022-01-05,2022-04-04,New Customers,0.099
48,49,Paid Search,Reactivation,2021-04-09,2021-05-24,Deal Seekers,0.128
49,50,Email,Retention,2021-02-14,2021-03-24,Churn Risk,0.023
50,0,No Campaign,Organic/Direct/Unattributed,NaT,NaT,Unknown,0.000


In [60]:
transactions_enriched = (
    transactions
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(campaigns_extended, on="campaign_id", how="left")
)

transactions_enriched.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag,signup_date,...,brand,base_price,launch_date,is_premium,channel,objective,start_date,end_date,target_segment,expected_uplift
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0,2021-11-26,...,Brand_76,14.58,2022-11-02,0.0,No Campaign,Organic/Direct/Unattributed,NaT,NaT,Unknown,0.000
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0,2021-06-24,...,Brand_20,58.26,2022-09-21,0.0,Email,Acquisition,2023-02-21,2023-03-27,Churn Risk,0.090
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0,2021-07-17,...,Brand_23,40.61,2021-04-24,0.0,Paid Search,Cross-sell,2022-10-20,2022-12-01,Deal Seekers,0.096
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0,2022-01-30,...,Brand_19,40.45,2021-05-26,0.0,Display,Reactivation,2022-03-19,2022-06-09,Churn Risk,0.063
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0,2023-06-15,...,Brand_39,14.64,2022-01-08,0.0,Display,Reactivation,2022-07-25,2022-10-07,Deal Seekers,0.111


In [61]:
valid_revenue_transactions = transactions_enriched[
    (transactions_enriched["refund_flag"] == 0) &
    (transactions_enriched["gross_revenue"].notna())
].copy()

total_transactions = transactions_enriched["transaction_id"].nunique()
valid_transactions_count = valid_revenue_transactions["transaction_id"].nunique()
unique_buyers = transactions_enriched["customer_id"].nunique()
gross_revenue = transactions_enriched["gross_revenue"].sum()
net_revenue = valid_revenue_transactions["gross_revenue"].sum()
refund_count = transactions_enriched["refund_flag"].sum()

summary_metrics = pd.DataFrame({
    "metric": [
        "Total Transactions",
        "Valid Revenue Transactions",
        "Unique Buyers",
        "Gross Revenue",
        "Net Revenue Excluding Refunds",
        "Refund Count"
    ],
    "value": [
        total_transactions,
        valid_transactions_count,
        unique_buyers,
        gross_revenue,
        net_revenue,
        refund_count
    ]
})

summary_metrics

,metric,value
0,Total Transactions,103127.00
1,Valid Revenue Transactions,89974.00
2,Unique Buyers,64035.00
3,Gross Revenue,8373966.36
4,Net Revenue Excluding Refunds,8630269.31
5,Refund Count,3029.00


In [62]:
funnel_order = ["view", "click", "add_to_cart", "purchase"]

funnel_df = (
    events[events["event_type"].isin(funnel_order)]
    .groupby("event_type")
    .size()
    .reindex(funnel_order)
    .reset_index(name="event_count")
)

funnel_df["previous_stage_count"] = funnel_df["event_count"].shift(1)

funnel_df["conversion_from_previous_percent"] = (
    funnel_df["event_count"] / funnel_df["previous_stage_count"] * 100
).round(2)

funnel_df["dropoff_count"] = (
    funnel_df["previous_stage_count"] - funnel_df["event_count"]
)

funnel_df

,event_type,event_count,previous_stage_count,conversion_from_previous_percent,dropoff_count
0,view,1043573,NaN,NaN,NaN
1,click,379008,1043573.0,36.32,664565.0
2,add_to_cart,284370,379008.0,75.03,94638.0
3,purchase,103127,284370.0,36.27,181243.0


In [63]:
revenue_by_channel = (
    valid_revenue_transactions
    .groupby("channel", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

revenue_by_channel

,channel,net_revenue
0,No Campaign,1744720.43
1,Affiliate,1652899.35
2,Paid Search,1581814.72
3,Email,1398378.96
4,Display,1248268.04
5,Social,1004187.81


In [64]:
revenue_by_category = (
    valid_revenue_transactions
    .groupby("category", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

revenue_by_category

,category,net_revenue
0,Electronics,3554175.61
1,Home,2053954.96
2,Fashion,1338781.78
3,Sports,1000708.47
4,Beauty,381439.05
5,Grocery,301209.44


In [65]:
revenue_by_loyalty = (
    valid_revenue_transactions
    .groupby("loyalty_tier", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

revenue_by_loyalty

,loyalty_tier,net_revenue
0,Bronze,4615872.65
1,Silver,2369478.75
2,Gold,1313263.06
3,Platinum,331654.85


In [69]:
"touchpoints_sample" in globals()

False

In [70]:
# WEEK 2 ATTRIBUTION LOGIC - SIMPLE FULL VERSION

# Step 1: Create No Campaign row
no_campaign_row = pd.DataFrame([{
    "campaign_id": 0,
    "channel": "No Campaign",
    "objective": "Organic/Direct/Unattributed",
    "start_date": pd.NaT,
    "end_date": pd.NaT,
    "target_segment": "Unknown",
    "expected_uplift": 0
}])

campaigns_extended = pd.concat([campaigns, no_campaign_row], ignore_index=True)

# Step 2: Join events with campaign details
events_enriched = events.merge(
    campaigns_extended,
    on="campaign_id",
    how="left"
)

# Step 3: Keep only valid transactions
valid_transactions = transactions[
    (transactions["refund_flag"] == 0) &
    (transactions["gross_revenue"].notna())
].copy()

valid_transactions = valid_transactions.rename(columns={
    "timestamp": "transaction_time",
    "campaign_id": "transaction_campaign_id"
})

# Step 4: Keep only useful event touchpoints
touchpoint_events = events_enriched[
    events_enriched["event_type"].isin(["view", "click", "add_to_cart"])
].copy()

# Step 5: Use only 5000 transactions first, because the data is large
sample_transactions = valid_transactions.sample(
    n=min(5000, len(valid_transactions)),
    random_state=42
)

# Step 6: Join customer touchpoints with transactions
touchpoints_sample = sample_transactions.merge(
    touchpoint_events,
    on="customer_id",
    how="left"
)

# Step 7: Keep only events before purchase
touchpoints_sample = touchpoints_sample[
    touchpoints_sample["timestamp"] <= touchpoints_sample["transaction_time"]
].copy()

# Step 8: Keep only events within 30 days before purchase
touchpoints_sample = touchpoints_sample[
    touchpoints_sample["timestamp"] >= (
        touchpoints_sample["transaction_time"] - pd.Timedelta(days=30)
    )
].copy()

# Step 9: Order touchpoints from first to last
touchpoints_sample = touchpoints_sample.sort_values(
    ["transaction_id", "timestamp"]
).copy()

touchpoints_sample["touch_order_asc"] = (
    touchpoints_sample.groupby("transaction_id").cumcount() + 1
)

# Step 10: Order touchpoints from last to first
touchpoints_sample = touchpoints_sample.sort_values(
    ["transaction_id", "timestamp"],
    ascending=[True, False]
).copy()

touchpoints_sample["touch_order_desc"] = (
    touchpoints_sample.groupby("transaction_id").cumcount() + 1
)

# Step 11: Count total touchpoints for each transaction
touchpoints_sample["total_touches"] = (
    touchpoints_sample.groupby("transaction_id")["event_id"].transform("count")
)

# Step 12: Calculate first-touch, last-touch, and linear revenue
touchpoints_sample["first_touch_revenue"] = np.where(
    touchpoints_sample["touch_order_asc"] == 1,
    touchpoints_sample["gross_revenue"],
    0
)

touchpoints_sample["last_touch_revenue"] = np.where(
    touchpoints_sample["touch_order_desc"] == 1,
    touchpoints_sample["gross_revenue"],
    0
)

touchpoints_sample["linear_revenue"] = (
    touchpoints_sample["gross_revenue"] / touchpoints_sample["total_touches"]
)

print("touchpoints_sample created successfully")
print("Rows:", touchpoints_sample.shape[0])
print("Transactions with touchpoints:", touchpoints_sample["transaction_id"].nunique())

touchpoints_sample.head()

touchpoints_sample created successfully
Rows: 2287
Transactions with touchpoints: 1827


,transaction_id,transaction_time,customer_id,product_id_x,quantity,discount_applied,gross_revenue,transaction_campaign_id,refund_flag,event_id,...,start_date,end_date,target_segment,expected_uplift,touch_order_asc,touch_order_desc,total_touches,first_touch_revenue,last_touch_revenue,linear_revenue
24642,168,2023-11-20 15:39:55,14983,499.0,1,0.0,89.69,8,0,526947,...,2022-09-20,2022-09-29,High Value,0.092,1,1,1,89.69,89.69,89.69
34315,249,2021-01-19 12:54:03,37776,1933.0,1,0.2,22.54,32,0,1197465,...,NaT,NaT,Unknown,0.000,1,1,1,22.54,22.54,22.54
10377,305,2021-11-12 17:45:29,74942,631.0,4,0.0,144.56,17,0,1306742,...,2023-07-02,2023-08-10,High Value,0.136,2,1,2,0.00,144.56,72.28
10379,305,2021-11-12 17:45:29,74942,631.0,4,0.0,144.56,17,0,1423782,...,2022-07-25,2022-09-07,New Customers,0.105,1,2,2,144.56,0.00,72.28
68883,371,2022-06-07 13:33:16,87054,230.0,1,0.0,94.61,7,0,1763466,...,2021-12-26,2022-02-26,New Customers,0.140,1,1,1,94.61,94.61,94.61


In [71]:
attribution_by_channel = (
    touchpoints_sample
    .groupby("channel", dropna=False)[[
        "first_touch_revenue",
        "last_touch_revenue",
        "linear_revenue"
    ]]
    .sum()
    .sort_values("linear_revenue", ascending=False)
    .reset_index()
)

attribution_by_channel

,channel,first_touch_revenue,last_touch_revenue,linear_revenue
0,No Campaign,97154.58,98729.80,98357.303000
1,Email,19363.93,16936.95,18149.544833
2,Affiliate,17232.95,17425.29,17208.536667
3,Paid Search,15965.53,16387.58,16101.316167
4,Display,15174.13,16329.29,15722.986667
5,Social,13874.17,12956.38,13225.602667


In [72]:
    attribution_by_campaign = (
        touchpoints_sample
        .groupby(["campaign_id", "channel"], dropna=False)[[
            "first_touch_revenue",
            "last_touch_revenue",
            "linear_revenue"
        ]]
        .sum()
        .sort_values("linear_revenue", ascending=False)
        .reset_index()
    )

    attribution_by_campaign.head(15)

,campaign_id,channel,first_touch_revenue,last_touch_revenue,linear_revenue
0,0,No Campaign,97154.58,98729.80,98357.303000
1,21,Email,3566.67,2410.42,2990.623000
2,19,Affiliate,2927.61,2961.09,2890.408333
3,30,Social,3064.00,2218.46,2591.913333
4,4,Display,2679.63,2469.00,2574.315000
5,17,Display,2247.03,2785.64,2546.231667
6,37,Paid Search,2578.53,2164.08,2404.765000
7,43,Paid Search,1620.90,2886.82,2253.860000
8,34,Email,2261.81,1898.73,2059.555000
9,24,Display,1950.08,2091.87,2027.328333


## Attribution Output Interpretation

The attribution model was successfully tested using a sample of valid transactions.

The output shows attributed revenue by campaign and channel under three attribution models:

- First-touch attribution gives full revenue credit to the first touchpoint before purchase.
- Last-touch attribution gives full revenue credit to the last touchpoint before purchase.
- Linear attribution divides revenue equally across all touchpoints before purchase.

The result shows that "No Campaign" receives the highest attributed revenue. This means many customer journeys are linked to organic, direct, or unattributed activity where campaign_id = 0.

Campaign-level results can be used later in the dashboard to compare which campaigns and channels contribute most under different attribution models.

In [73]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

attribution_by_channel.to_csv(
    OUTPUT_DIR / "attribution_by_channel.csv",
    index=False
)

attribution_by_campaign.to_csv(
    OUTPUT_DIR / "attribution_by_campaign.csv",
    index=False
)

print("Attribution output files saved successfully.")

Attribution output files saved successfully.


# Week 3: Dashboard Metric Summary Tables